In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

# Go from notebooks/ to the project root
project_root = Path.cwd().parent

monday_path = (
    project_root
    / "data"
    / "raw"
    / "traffic_labels"
    / "Monday-WorkingHours.pcap_ISCX.csv.parquet"
)

portscan_path = (
    project_root
    / "data"
    / "raw"
    / "traffic_labels"
    / "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet"
)

# Check that the files actually exist
print("Project root:", project_root)
print("Monday file exists:", monday_path.exists())
print("PortScan file exists:", portscan_path.exists())

# Load the data
monday_df = pd.read_parquet(monday_path)
portscan_df = pd.read_parquet(portscan_path)

# Explore the datasets
for name, df in [
    ("monday_df", monday_df),
    ("portscan_df", portscan_df)
]:
    print(f"{name} shape: {df.shape}")

    print(f"\n{name} first 5 rows:")
    print(df.head())

    print(f"\n{name} columns:")
    print(list(df.columns))

    print(f"\n{name} dtypes:")
    print(df.dtypes)

    print("\n" + "-" * 80 + "\n")

# Task 2: read-only structure and label analysis for both datasets
for name, df in [("monday_df", monday_df), ("portscan_df", portscan_df)]:
    print(f"\n===== {name.upper()} ANALYSIS =====")

    print("Number of rows and columns:")
    print(df.shape)

    if "Label" in df.columns:
        label_counts = df["Label"].value_counts(dropna=False)
        label_percents = df["Label"].value_counts(normalize=True, dropna=False) * 100
        print("\nLabel value counts:")
        print(pd.concat([label_counts, label_percents], axis=1, keys=["count", "percentage"]))
    else:
        print("\nLabel column not found.")

    ts_col = "Timestamp" if "Timestamp" in df.columns else None
    if ts_col is not None:
        print("\nTimestamp dtype:")
        print(df[ts_col].dtype)

        print("Timestamp min:")
        print(df[ts_col].min())

        print("Timestamp max:")
        print(df[ts_col].max())

        print("Number of unique timestamps:")
        print(df[ts_col].nunique())
    else:
        print("\nTimestamp column not found.")

    print("\nNumber of unique source IPs:")
    print(df["Source IP"].nunique() if "Source IP" in df.columns else "Column not found")

    print("Number of unique destination IPs:")
    print(df["Destination IP"].nunique() if "Destination IP" in df.columns else "Column not found")

    print("Number of unique source ports:")
    print(df["Source Port"].nunique() if "Source Port" in df.columns else "Column not found")

    print("Number of unique destination ports:")
    print(df["Destination Port"].nunique() if "Destination Port" in df.columns else "Column not found")

    print("Number of unique Flow IDs:")
    print(df["Flow ID"].nunique() if "Flow ID" in df.columns else "Column not found")

    print("\nDuplicate row count:")
    print(df.duplicated().sum())

    print("\n" + "-" * 80 + "\n")

print("Task 2 analysis complete.")


Project root: d:\AAA_SIH\MVP\ai-network-attack-forecasting
Monday file exists: True
PortScan file exists: True
monday_df shape: (529918, 85)

monday_df first 5 rows:
                                  Flow ID      Source IP  Source Port  \
0   192.168.10.5-8.254.250.126-49188-80-6  8.254.250.126           80   
1   192.168.10.5-8.254.250.126-49188-80-6  8.254.250.126           80   
2   192.168.10.5-8.254.250.126-49188-80-6  8.254.250.126           80   
3   192.168.10.5-8.254.250.126-49188-80-6  8.254.250.126           80   
4  192.168.10.14-8.253.185.121-49486-80-6  8.253.185.121           80   

  Destination IP  Destination Port  Protocol           Timestamp  \
0   192.168.10.5             49188         6 2017-07-03 11:55:58   
1   192.168.10.5             49188         6 2017-07-03 11:55:58   
2   192.168.10.5             49188         6 2017-07-03 11:55:58   
3   192.168.10.5             49188         6 2017-07-03 11:55:58   
4  192.168.10.14             49486         6 2017-07-03

In [ ]:
def diagnostic_report(df, name):
    print(f"\n===== {name} DATA QUALITY DIAGNOSTIC REPORT =====")

    missing_by_col = df.isnull().sum()
    nan_by_col = df.isna().sum()

    print("\n1) Missing values per column:")
    print(missing_by_col)

    print("\n2) NaN values per column:")
    print(nan_by_col)

    inf_pos_by_col = {}
    inf_neg_by_col = {}
    for col in df.columns:
        s = pd.to_numeric(df[col], errors='coerce')
        inf_pos_by_col[col] = int(np.isposinf(s.to_numpy(dtype=float, na_value=np.nan)).sum())
        inf_neg_by_col[col] = int(np.isneginf(s.to_numpy(dtype=float, na_value=np.nan)).sum())

    inf_pos_df = pd.Series(inf_pos_by_col)
    inf_neg_df = pd.Series(inf_neg_by_col)

    print("\n3) Positive infinity values per column:")
    print(inf_pos_df)

    print("\n3) Negative infinity values per column:")
    print(inf_neg_df)

    problematic_cols = set(
        missing_by_col[missing_by_col > 0].index.tolist()
    )
    problematic_cols |= set(
        inf_pos_df[inf_pos_df > 0].index.tolist()
    )
    problematic_cols |= set(
        inf_neg_df[inf_neg_df > 0].index.tolist()
    )

    print("\n4) Columns containing any problematic values:")
    if problematic_cols:
        print(sorted(problematic_cols))
    else:
        print("No problematic columns found.")

    numeric_df = df.select_dtypes(include=[np.number])
    if not numeric_df.empty:
        print("\n5) Numeric summary statistics:")
        numeric_summary = numeric_df.agg(["min", "max", "mean", "std"])
        print(numeric_summary)

        const_cols = numeric_df.columns[numeric_df.nunique(dropna=True).le(1)].tolist()
        print("\n6) Numeric columns with zero variance / constant values:")
        if const_cols:
            print(const_cols)
        else:
            print("No constant numeric columns found.")

        negative_cols = numeric_df.columns[numeric_df.lt(0).any()].tolist()
        print("\n7) Numeric columns containing negative values:")
        if negative_cols:
            print(negative_cols)
        else:
            print("No numeric columns contain negative values.")

        abs_max_vals = numeric_df.apply(lambda s: abs(s).max())
        top_abs_max = abs_max_vals.sort_values(ascending=False).head(15)
        print("\n8) 15 numeric columns with the largest absolute maximum values:")
        print(top_abs_max)
    else:
        print("\n5-8) No numeric columns available for summary statistics.")

    print("\n" + "-" * 80)


# Read-only data quality diagnostic for both datasets
for dataset_name, dataset_df in [("monday_df", monday_df), ("portscan_df", portscan_df)]:
    diagnostic_report(dataset_df, dataset_name)

print("\nDiagnostic report complete. No data was modified.")
